Why Multi-Agent Systems

A single agent with many tools becomes brittle at scale.

* Too many tools confuses the LLM(it picks the wrong one)
* A single context window handles everything - no specialization
* One failure affects the whole pipeline.
* Hard to test, debug, or improve individual capabilities.

Multi-agent Systems solve this by decomposing work into sepcialized agents each with a:
* focused set of tools.
* tuned system prompt.
* the right llm for its task
* clear input/output contracts 

LangGraph is purpose-built for multi-agent orchestration — it lets you wire agents together as graph nodes, pass state between them, and build hierarchical systems that mirror how real teams work.


**Multi-Agent-System**

A multi-agent system (MAS) is a workflow where multiple AI Agents collaborate to solve a problem.

instead of : ```user-->one agent-->answer``` 

we have ```user-->agent1-->agent2-->agent3-->Final answer``` Each agent has a final responsibility.


**Overall Architecture**
``` markdown
                  User
                    │
                    ▼
             Supervisor Agent
        ┌─────────┼─────────┐
        ▼         ▼         ▼
 Search Agent SQL Agent Python Agent
        │         │         │
        └─────────┼─────────┘
                  ▼
            Final Response
```

Supervisor coordinate everything.


#### Components 

1. Multiple agents
2. Agent communication
3. Supervisor pattern
4. worker pattern
5. Router pattern
6. Collaboration
7. Hierachical Agents 

**Core Concepts Before Code**

``` markdown
Single Agent                    Multi-Agent System 
─────────────────────           ──────────────────────────────
One LLM                         Many specialized LLMs
All tools in one place          Each agent has its own tools
One system prompt               Per-agent prompts
Single context window           Isolated contexts per agent
Sequential tool calls           Parallel or routed execution
```

Three fundamental patterns:

``` markdown
1. SUPERVISOR PATTERN           2. WORKER PATTERN            3. ROUTER PATTERN
   Supervisor                      Orchestrator                 Router
   /    |    \                      /    |    \                  /   |   \
 A1    A2    A3                  W1    W2    W3               A1   A2   A3
 
 Supervisor routes              Workers run in               Router picks ONE
 and aggregates                 parallel, merge              agent per request
```

Setup

In [3]:
from langgraph.graph import StateGraph,MessagesState,END,START # type:ignore
from langgraph.prebuilt import create_react_agent, ToolNode,tools_condition # type:ignore
from langchain_groq import ChatGroq # type:ignore
from langchain_google_genai import ChatGoogleGenerativeAI # type:ignore
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage # type:ignore
from langchain_core.tools import tool # type:ignore
from typing import TypedDict, Annotated,List, Literal,Sequence
import operator

Part 1 — Multiple Agents (Foundations)

1.1 What Is an Agent in LangGraph

An agent is just a compiled graph (or subgraph) that:
* takes messages as input
* runs an llm with tools in the loop
* returns a final message

In [4]:
### simple agent 

# step 1 : import the necessary libraries required from langgraph and langchain

import os
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI # type:ignore
from langchain_core.tools import tool # type:ignore # to create a tool
from langchain_core.messages import HumanMessage # type:ignore  # To create user messages
from langgraph.prebuilt import create_react_agent  # type:ignore     # To create the agent

# step 2: print to confirm env vars loaded correctly 
print("Project  :", os.getenv("GOOGLE_CLOUD_PROJECT"))
print("Location :", os.getenv("GOOGLE_CLOUD_LOCATION"))
print("VertexAI :", os.getenv("GOOGLE_GENAI_USE_VERTEXAI"))

# step 3: Define Tools

@tool
def search_products(query: str = "") -> str:
    """
    Search the product catalogue.
    - Pass a keyword like 'laptop' to search for a specific product.
    - Pass empty string or 'all' to list ALL available products.
    
    Args:
        query: Product keyword to search. Leave empty to list all products.
    """
    catalogue = {
        "laptop": "Dell XPS 15 — ₹85,000",
        "phone":  "Samsung Galaxy S24 — ₹60,000",
        "tablet": "iPad Air — ₹55,000"
    }
    
    if query == "" or query == "all" or query == "*" or query == "everything":
        all_poducts = []
        for key, value in catalogue.items():
            all_poducts.append("-"+value)
        # Join all products into one big string with a newline between each
        result_string = "\n".join(all_poducts)
    
    return "Availbale products:\n" + result_string


@tool
def check_stock(product_name: str) -> str:
    """
    Check how many units of a product are in stock.
    Args:
        product_name: Name of the product (e.g. 'laptop', 'phone', 'tablet')
    """
    stock = {
        "laptop": 12,
        "phone":  45,
        "tablet": 8
    }
    
    qty = stock.get(product_name.lower(), 0)
    
    if qty > 0:
        return f"{product_name}: {qty} units in stock"
    else:
        return f"{product_name}: Out of stock"

# step 4 : Create the gemini model
llm = ChatGoogleGenerativeAI(
    model = "gemini-1.5-flash",
    project = os.getenv("GOOGLE_CLOUD_PROJECT"),
    location = os.getenv("GOOGLE_CLOUD_LOCATION"),
    temperature = 0,
)
print("\n Model Created Successfully")


# step 5: Create Agent
product_agent = create_react_agent(
    model = llm,
    tools = [search_products,check_stock],
    prompt = (
        "You are a helpful product catalog specialist. "
        "Help users find products and check stock availability. "
        "Always use the available tools to answer questions."
    )
)
print("Agent Created Successfully")

# step 6: Run Agent
print("\n--- Running Agent ---\n")
result = product_agent.invoke(
    {"messages":[
        HumanMessage(content = " i want to see what are all the products are available")
    ]
    }
)

print("Agent Response:")
print(result["messages"][-1].content)

last_message = result["messages"][-1].content_block

# content is a list of blocks - loop and find the text block.
for block in last_message:
    if block["type"] == "text":
        print(block["text"])


Project  : multi-agent-systems-502406
Location : us-central1
VertexAI : True

 Model Created Successfully
Agent Created Successfully

--- Running Agent ---



C:\Users\subramani.v\AppData\Local\Temp\ipykernel_16672\370512994.py:77: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  product_agent = create_react_agent(


ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/multi-agent-systems-502406/locations/us-central1/publishers/google/models/gemini-1.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.', 'status': 'NOT_FOUND'}}